# RAG: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

Two retrieval pipelines are compared on the **same chunked corpus**: the EU General Data Protection Regulation (GDPR, Regulation 2016/679, 88 pages).

| | Pipeline 1 – Baseline | Pipeline 2 – Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder → FAISS top 10 | same top 10 |
| Stage 2 | take top 3 as-is | Cross-encoder re-scores all 10 → top 3 |
| Generation | Gemini, shared prompt | Gemini, same prompt |

**Run:** `py -3.12 -m jupyter notebook rag_baseline_vs_rerank.ipynb`. Needs `GEMINI_KEYS` (comma-separated) or `GEMINI_API_KEY` in a `.env` file and `data/gdpr.pdf`.

## 0. Setup

In [1]:
# py -3.12 -m pip install pymupdf sentence-transformers faiss-cpu google-genai python-dotenv
import os, re, time, statistics
import fitz                      # PyMuPDF, for PDF text extraction
import faiss                     # vector store
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()                    # reads GEMINI_KEYS / GEMINI_API_KEY from .env
pd.set_option("display.max_colwidth", 120)

PDF_PATH   = "data/gdpr.pdf"
BI_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
CE_MODEL   = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL  = "gemini-3.5-flash-lite"   # free tier: ~20 requests/day per model
CHUNK_SIZE, CHUNK_OVERLAP = 800, 150   # characters
TOP_K, TOP_N = 10, 3                   # retrieve 10, give the LLM 3

C:\Users\amita\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load and chunk the PDF
Text is extracted page by page and the whitespace is normalised. Each page is cut into fixed-size, overlapping character windows. The overlap keeps a sentence that crosses a boundary intact in at least one chunk. Every chunk keeps its page number for citations. **Both pipelines use exactly this list.**

In [2]:
doc = fitz.open(PDF_PATH)
pages = [re.sub(r"\s+", " ", page.get_text()) for page in doc]

chunks = []
for page_no, text in enumerate(pages, start=1):
    for start in range(0, len(text), CHUNK_SIZE - CHUNK_OVERLAP):
        piece = text[start:start + CHUNK_SIZE].strip()
        if len(piece) > 100:                       # drop tiny page-end fragments
            chunks.append({"id": len(chunks), "page": page_no, "text": piece})

print(f"{len(doc)} pages -> {len(chunks)} chunks")
chunks[300]

88 pages -> 580 chunks


{'id': 300,
 'page': 39,
 'text': 'igation of professional secrecy under Union or Member State law or rules established by national competent bodies or by another person also subject to an obligation of secrecy under Union or Member State law or rules established by national competent bodies. 4. Member States may maintain or introduce further conditions, including limitations, with regard to the processing of genetic data, biometric data or data concerning health. Article 10 Processing of personal data relating to criminal convictions and offences Processing of personal data relating to criminal convictions and offences or related security measures based on Article 6(1) shall be carried out only under the control of official authority or when the processing is authorised by Union or Member State law providing for appropriat'}

## 2. Pipeline 1, Stage 1: Bi-encoder + FAISS
The bi-encoder embeds every chunk **independently** into a 384-d vector, once, offline. The vectors are L2-normalised, so the inner product in `IndexFlatIP` equals cosine similarity. At query time only the query is embedded.

In [3]:
bi_encoder = SentenceTransformer(BI_MODEL)

chunk_emb = bi_encoder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                              batch_size=64, show_progress_bar=True)
index = faiss.IndexFlatIP(chunk_emb.shape[1])   # exact inner-product search
index.add(chunk_emb)
print("indexed:", index.ntotal, "vectors of dim", chunk_emb.shape[1])

def retrieve(query, k=TOP_K):
    """Return the top-k chunks by cosine similarity: [(chunk_id, score), ...]"""
    q = bi_encoder.encode([query], normalize_embeddings=True)
    scores, ids = index.search(q, k)
    return list(zip(ids[0].tolist(), scores[0].tolist()))

retrieve("right to erasure")[:3]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10289.46it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:  10%|█         | 1/10 [00:00<00:02,  3.83it/s]

Batches:  30%|███       | 3/10 [00:00<00:00,  7.01it/s]

...

Batches:  70%|███████   | 7/10 [00:00<00:00,  9.08it/s]

Batches:  80%|████████  | 8/10 [00:00<00:00,  9.13it/s]

Batches: 100%|██████████| 10/10 [00:01<00:00,  9.44it/s]

indexed: 580 vectors of dim 384


[(326, 0.5708491802215576),
 (100, 0.5610976219177246),
 (327, 0.5276192426681519)]

## 3. Pipeline 2, Stage 2: Cross-encoder re-ranking
The cross-encoder reads each `(query, chunk)` pair **together** in one transformer pass and outputs a relevance score. Only the 10 baseline candidates get re-scored.

In [4]:
cross_encoder = CrossEncoder(CE_MODEL)

def rerank(query, candidates):
    """Re-sort bi-encoder candidates by cross-encoder score: [(chunk_id, ce_score), ...]"""
    ids = [cid for cid, _ in candidates]
    scores = cross_encoder.predict([(query, chunks[cid]["text"]) for cid in ids])
    return sorted(zip(ids, scores.tolist()), key=lambda x: -x[1])

rerank("right to erasure", retrieve("right to erasure"))[:3]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8071.44it/s]

[(100, 7.261177062988281), (326, 7.225945472717285), (327, 6.1185526847839355)]

## 4. Generation: one shared prompt for both pipelines
Only the 3 context chunks differ between Answer 1 and Answer 2. Temperature 0 keeps the comparison deterministic.

In [5]:
# GEMINI_KEYS (comma-separated) + GEMINI_API_KEY, de-duplicated; answer() rotates to the next key on a quota error
raw = os.getenv("GEMINI_KEYS", "") + "," + os.getenv("GEMINI_API_KEY", "")
API_KEYS = list(dict.fromkeys(k.strip().strip("\"'") for k in raw.split(",") if k.strip()))
clients = [genai.Client(api_key=k, http_options=types.HttpOptions(timeout=90_000)) for k in API_KEYS]
key_idx = 0
llm_times = []   # LLM latency per successful call, reused in the latency section
print(f"{len(clients)} Gemini key(s) loaded")

PROMPT = """You are answering questions about the EU GDPR.
Answer ONLY from the context below. If the context does not contain the answer, say so.
Mention any exceptions or conditions that appear in the context. Cite pages like [p.12].

Context:
{context}

Question: {question}
Answer (max 120 words):"""

def answer(query, chunk_ids):
    global key_idx
    context = "\n\n".join(f"[p.{chunks[i]['page']}] {chunks[i]['text']}" for i in chunk_ids)
    for attempt in range(3 * len(clients) + 3):
        try:
            t = time.perf_counter()
            resp = clients[key_idx].models.generate_content(
                model=LLM_MODEL,
                contents=PROMPT.format(context=context, question=query),
                config=types.GenerateContentConfig(temperature=0),
            )
            llm_times.append(time.perf_counter() - t)   # only successful calls count towards latency
            return resp.text.strip()
        except genai.errors.APIError as e:
            if e.code not in (429, 503):                # 429 = free-tier rate limit, 503 = overloaded
                raise
            if e.code == 429 and len(clients) > 1:      # quota used up on this key -> next key
                key_idx = (key_idx + 1) % len(clients)
                print(f"  (429, switching to key #{key_idx + 1})")
                if key_idx != 0:
                    continue                            # untried key: retry immediately
            print(f"  ({e.code}, waiting 20 s and retrying…)")
            time.sleep(20)
    return "[LLM call failed after retries]"

8 Gemini key(s) loaded


## 5. Side-by-side runner
`run_both` executes both pipelines on one query and shows:
- a **rank shift table** for all 10 candidates (bi-encoder rank → cross-encoder rank). ★ marks the chunk containing the key clause, found with a text `marker`.
- **Answer 1** (baseline top 3) and **Answer 2** (re-ranked top 3).

In [6]:
def run_both(query, marker=None):
    cands = retrieve(query)
    reranked = rerank(query, cands)
    ce_rank = {cid: r for r, (cid, _) in enumerate(reranked, 1)}
    ce_score = dict(reranked)

    rows = [{
        "key": "★" if marker and marker.lower() in chunks[cid]["text"].lower() else "",
        "chunk": cid, "page": chunks[cid]["page"],
        "bi_rank": r, "bi_score": round(s, 3),
        "ce_rank": ce_rank[cid], "ce_score": round(ce_score[cid], 2),
        "shift": f"#{r} → #{ce_rank[cid]}",
        "snippet": chunks[cid]["text"][:110] + "…",
    } for r, (cid, s) in enumerate(cands, 1)]
    display(pd.DataFrame(rows).set_index("bi_rank"))

    top3_base = [cid for cid, _ in cands[:TOP_N]]
    top3_rr   = [cid for cid, _ in reranked[:TOP_N]]
    print("Baseline top-3 :", top3_base, "\nRe-ranked top-3:", top3_rr)

    for label, ids in [("ANSWER 1 (baseline)", top3_base), ("ANSWER 2 (re-ranked)", top3_rr)]:
        print(f"\n=== {label} ===\n{answer(query, ids)}")

## 6. Failure-mode analysis (stress tests)
These queries were picked by screening about 20 candidates with retrieval + re-ranking only, and keeping the cases where the chunk with the decisive clause sits at rank 4–10 in the baseline.

| # | Category | Query | Key clause |
|---|---|---|---|
| 1 | Keyword trap / distractor | *Can my employer monitor me at work?* | Art. 88 says "employment context", while "monitoring" pulls in supervisory-authority chunks |
| 2 | Exception / corner case | *Can I get my personal data deleted, and when can the company refuse?* | Recital 65: retention lawful for freedom of expression, legal obligations, etc. |
| 3 | Negation / constraint | *When does a company not need to tell affected people about a breach?* | Art. 34(3): communication "shall not be required" if… |
| 4 | Buried chunk (rank 4–10) | *Must every company appoint a data protection officer?* | Art. 37(1): only public bodies or large-scale core monitoring / special data |
| 5 | Control (no failure) | *Can I take my data to another service provider?* | Recital 68 restates the Art. 20 portability rule and is already in the baseline top 3, so both answers should match |

In [7]:
# (category, query, text marker identifying the key chunk)
FAILURE_QUERIES = [
    ('Keyword trap', 'Can my employer monitor me at work?', 'employment context'),
    ('Exception', 'Can I get my personal data deleted, and when can the company refuse?', 'freedom of expression'),
    ('Negation', 'When does a company not need to tell affected people about a breach?', 'shall not be required'),
    ('Buried chunk', 'Must every company appoint a data protection officer?', 'core activities of the controller or the processor consist'),
    ('Control', 'Can I take my data to another service provider?', 'interoperable format'),
]

### 6.1 Keyword trap
**Query:** Can my employer monitor me at work?

In [8]:
run_both('Can my employer monitor me at work?', marker='employment context')

,key,chunk,page,bi_score,ce_rank,ce_score,shift,snippet
bi_rank,,,,,,,,
1,,195,24,0.367,6,-10.28,#1 → #6,"nsistent monitoring and enforcement of this Regulation throughout the Union, the supervisory authorities shoul…"
2,,196,24,0.350,9,-11.10,#2 → #9,"a ban, on processing. Member States may specify other tasks related to the protection of personal data under t…"
3,★,562,84,0.349,1,-5.76,#3 → #1,tion of the rights and freedoms in respect of the processing of employees' personal data in the employment con…
4,,474,69,0.342,5,-10.12,#4 → #5,(p) draft and publish the criteria for accreditation of a body for monitoring codes of conduct pursuant to Art…
5,,471,68,0.341,8,-10.51,#5 → #8,"omplainant of the progress and the outcome of the investigation within a reasonable period, in particular if f…"
6,,397,56,0.336,2,-9.20,#6 → #2,llowing tasks: (a) to inform and advise the controller or the processor and the employees who carry out proces…
7,,476,69,0.333,10,-11.21,#7 → #10,"ve costs, or refuse to act on the request. The supervisory authority shall bear the burden of demonstrating th…"
8,★,236,29,0.331,3,-9.65,#8 → #3,"processing of personal data under the provisions of Union and Member State law, and in particular does not alt…"
9,,456,65,0.325,7,-10.31,#9 → #7,Independent supervisory authorities Section 1 Independent status Article 51 Supervisory authority 1. Each Memb…


Baseline top-3 : [195, 196, 562] 
Re-ranked top-3: [562, 397, 236]



=== ANSWER 1 (baseline) ===
Based on the provided context, rules concerning the processing of employees' personal data in the employment context—including management and planning and organisation of work—may be provided for by law or collective agreements [p.84]. 

These rules are subject to the following conditions:
* They must include suitable and specific measures to safeguard the data subject's human dignity, legitimate interests, and fundamental rights [p.84]. 
* They can be used for purposes such as recruitment, the performance of the employment contract (including discharging obligations laid down by law or collective agreements), health and safety at work, equality and diversity, protection of employer or customer property, and the termination of the employment relationship [p.84].

The provided context does not explicitly mention direct "monitoring" of employees at work, but addresses data processing in the employment context.



=== ANSWER 2 (re-ranked) ===
Based on the provided context, rules may be provided by Member State law or collective agreements (including "works agreements") regarding the processing of employees' personal data in the employment context [p.29]. 

Furthermore, the processing of employees' personal data includes purposes such as the "protection of employer's or customer's property" and the "management, planning and organisation of work" [p.84]. However, these rules must include "suitable and specific measures to safeguard the data subject's human dignity, legitimate interests and fundamental rights" [p.84]. 

The provided context does not explicitly use the word "monitor," but it addresses the processing of employees' personal data and workplace rules under the conditions stated above [p.29, p.84].


**Observation (chunk 562: #3 → #1):** The baseline's top two results, chunks 195 and 196, discussed the monitoring and enforcement activities of supervisory authorities rather than an employer monitoring employees. The word "monitoring" created a strong topical match, even though the passages addressed a different question. Because the Article 88 chunk was already third, both answers were correct, but two of the three baseline passages were irrelevant. The baseline did not always produce a wrong answer, but its retrieval quality could still be improved.

### 6.2 Exception
**Query:** Can I get my personal data deleted, and when can the company refuse?

In [9]:
run_both('Can I get my personal data deleted, and when can the company refuse?', marker='freedom of expression')

,key,chunk,page,bi_score,ce_rank,ce_score,shift,snippet
bi_rank,,,,,,,,
1,,327,43,0.622,3,-1.60,#1 → #3,1. The data subject shall have the right to obtain from the controller the erasure of personal data concerning…
2,,97,12,0.586,4,-2.83,#2 → #4,being able to react to potential requests. (65) A data subject should have the right to have personal data con…
3,,328,44,0.554,5,-3.71,#3 → #5,(b) the data subject withdraws consent on which the processing is based according to point (a) of Article 6(1)…
4,,378,53,0.539,9,-5.54,#4 → #9,2. The communication to the data subject referred to in paragraph 1 of this Article shall describe in clear an…
5,,401,57,0.538,10,-8.15,#5 → #10,authorities and the communication of such personal data breaches to data subjects; (j) the transfer of persona…
6,★,99,13,0.535,1,-0.97,#6 → #1,"his or her consent as a child and is not fully aware of the risks involved by the processing, and later wants …"
7,★,329,44,0.534,6,-4.63,#7 → #6,ected in relation to the offer of information society services referred to in Article 8(1). 2. Where the contr…
8,,323,43,0.526,7,-5.11,#8 → #7,Article 15 Right of access by the data subject 1. The data subject shall have the right to obtain from the con…
9,,318,42,0.524,8,-5.19,#9 → #8,"t to withdraw consent at any time, without affecting the lawfulness of processing based on consent before its …"


Baseline top-3 : [327, 97, 328] 
Re-ranked top-3: [99, 54, 327]



=== ANSWER 1 (baseline) ===
Yes, you have the right to obtain from the controller the erasure of your personal data without undue delay if certain grounds apply [p.43]. 

The context specifies that you have a right to be forgotten or erased when:
- The data is no longer necessary for the purposes it was collected or processed [p.12, p.43];
- You withdraw your consent (and there is no other legal ground for processing) [p.12, p.44];
- You object to the processing, and there are no overriding legitimate grounds for the processing (or you object pursuant to Article 21(2)) [p.12, p.44];
- The data retention infringes the Regulation, Union, or Member State law, or the processing otherwise does not comply with the Regulation [p.12];
- The personal data have been unlawfully processed [p.44];
- The data must be erased to comply with a legal obligation in Union or Member State law [p.44];
- The data was collected in relation to the offer of information society services [p.44].

The provided co


=== ANSWER 2 (re-ranked) ===
Yes, you have the right to obtain the erasure of your personal data without undue delay when one of the following grounds applies: the personal data are no longer necessary in relation to the purposes for which they were collected or otherwise processed [p.43]. You can also exercise this right to remove personal data given as a child, even if you are no longer a child [p.13]. 

However, the further retention of personal data remains lawful and the company can refuse erasure where it is necessary for:
* Exercising the right of freedom of expression and information [p.13]
* Compliance with a legal obligation [p.13]
* The performance of a task carried out in the public interest or in the exercise of official authority vested in the controller [p.13]
* Grounds of public interest in the area of public health [p.13]
* Archiving purposes in the public interest, scientific or historical research purposes, or statistical purposes [p.13]


**Observation (chunk 99: #6 → #1):** The baseline's top three results focused on when personal data can be erased (Article 17(1) and the beginning of Recital 65). They matched the deletion part of the question but not the refusal part, so Answer 1 left out the refusal conditions entirely. After re-ranking, the rest of Recital 65 reached first place and Answer 2 listed all five refusal grounds.

### 6.3 Negation
**Query:** When does a company not need to tell affected people about a breach?

In [10]:
run_both('When does a company not need to tell affected people about a breach?', marker='shall not be required')

,key,chunk,page,bi_score,ce_rank,ce_score,shift,snippet
bi_rank,,,,,,,,
1,,376,52,0.567,3,-4.04,#1 → #3,"l data breach, including, where appropriate, measures to mitigate its possible adverse effects. 4. Where, and …"
2,,137,17,0.567,5,-4.63,#2 → #5,"enforcement authorities. For example, the need to mitigate an immediate risk of damage would call for prompt c…"
3,,135,17,0.548,1,-3.42,#3 → #1,"aware that a personal data breach has occurred, the controller should notify the personal data breach to the s…"
4,,138,17,0.545,9,-5.83,#4 → #9,to account in particular the nature and gravity of the personal data breach and its consequences and adverse e…
5,,136,17,0.517,7,-4.82,#5 → #7,"communicate to the data subject a personal data breach, without undue delay, where that personal data breach i…"
6,★,378,53,0.513,2,-3.86,#6 → #2,2. The communication to the data subject referred to in paragraph 1 of this Article shall describe in clear an…
7,,375,52,0.510,8,-4.96,#7 → #8,hout undue delay after becoming aware of a personal data breach. 3. The notification referred to in paragraph …
8,,374,52,0.499,6,-4.73,#8 → #6,Member State law. Article 33 Notification of a personal data breach to the supervisory authority 1. In the cas…
9,,139,17,0.451,10,-6.61,#9 → #10,"forms of misuse. Moreover, such rules and procedures should take into account the legitimate interests of law-…"


Baseline top-3 : [376, 137, 135] 
Re-ranked top-3: [135, 378, 376]



=== ANSWER 1 (baseline) ===
Based on the provided context, the controller must communicate a personal data breach to the data subject without undue delay when the breach is "likely to result in a high risk to the rights and freedoms of natural persons" [p.52, p.17]. 

The context does not explicitly list circumstances where a company *does not* need to tell affected people, other than the inverse condition: that communication to the data subject is required when there is a likely high risk to their rights and freedoms [p.52, p.17].



=== ANSWER 2 (re-ranked) ===
Based on the provided context, the communication of a personal data breach to the data subject is not required if any of the following conditions are met:

(a) The controller has implemented appropriate technical and organisational protection measures, and those measures were applied to the personal data affected by the breach, particularly those that render the personal data unintelligible to any unauthorized person, such as encryption [p.53].
(b) The controller has taken subsequent measures which ensure that the high risk to the rights and freedoms of data subjects is no longer likely to materialize [p.53–54 (implied context)]. 
*(Note: The context cuts off at "ensure that the", but lists conditions (a) and (b) under Article 34(3) [p.53]).*


**Observation (chunk 378: #6 → #2):** The baseline retrieved passages about reporting breaches to supervisory authorities (Article 33) and general breach recitals (85–87). These matched the topic but not the negation, so Answer 1 stated that the context did not list the exceptions. With Article 34(3) promoted to second, Answer 2 identified the encryption and subsequent-measures exceptions, but still missed the third exception because it fell outside the retrieved chunk.

### 6.4 Buried chunk
**Query:** Must every company appoint a data protection officer?

In [11]:
run_both('Must every company appoint a data protection officer?', marker='core activities of the controller or the processor consist')

,key,chunk,page,bi_score,ce_rank,ce_score,shift,snippet
bi_rank,,,,,,,,
1,,393,55,0.717,2,2.26,#1 → #2,"to in paragraph 1, the controller or processor or associations and other bodies representing categories of con…"
2,,396,56,0.656,4,1.22,#2 → #4,cessor. 4. Data subjects may contact the data protection officer with regard to all issues related to processi…
3,,395,56,0.620,5,0.84,#3 → #5,2. The controller and processor shall support the data protection officer in performing the tasks referred to …
4,,394,55,0.588,3,1.27,#4 → #3,"essor, or fulfil the tasks on the basis of a service contract. 7. The controller or the processor shall publis…"
5,★,391,55,0.581,7,0.41,#5 → #7,"in the public interest, including processing in relation to social protection and public health. Section 4 Dat…"
6,,534,79,0.579,9,-2.80,#6 → #9,n Data Protection Supervisor. 2. The secretariat shall perform its tasks exclusively under the instructions of…
7,★,392,55,0.571,1,3.07,#7 → #1,core activities of the controller or the processor consist of processing on a large scale of special categorie…
8,,151,19,0.571,6,0.82,#8 → #6,and the protection required for the personal data processed by the controller or the processor. Such data prot…
9,,164,20,0.565,8,-2.14,#9 → #8,"nsultations with it in order to remedy the situation. (108) In the absence of an adequacy decision, the contro…"


Baseline top-3 : [393, 396, 395] 
Re-ranked top-3: [392, 393, 394]



=== ANSWER 1 (baseline) ===
Based on the provided context, the controller or processor (or associations and other bodies representing them) **may** designate a data protection officer, or **shall** designate one where required by Union or Member State law [p.55].



=== ANSWER 2 (re-ranked) ===
Based on the provided context, the answer is not fully contained, as the context does not explicitly mention "every company." However, the context outlines the following rules and conditions regarding the designation of a data protection officer:

* A data protection officer must be designated if the core activities of the controller or processor consist of processing special categories of data on a large scale (pursuant to Article 9) and personal data relating to criminal convictions and offences (pursuant to Article 10) [p.55].
* In other cases, controllers, processors, associations, or bodies representing them *may* designate a data protection officer, or *shall* designate one where required by Union or Member State law [p.55]. 
* A group of undertakings may appoint a single data protection officer provided they are easily accessible from each establishment [p.55]. 
* A single data protection officer may be designated for several public authorities or b

**Observation (chunk 392: #7 → #1):** The baseline retrieved passages about DPO tasks and the "may designate" provision, which made the appointment appear optional. Once Article 37(1)(c) moved to first, Answer 2 explained the mandatory condition for large-scale processing of special categories of data. It was still incomplete, since the conditions in Article 37(1)(a) and (b) were not among the final three chunks.

### 6.5 Control
**Query:** Can I take my data to another service provider?

In [12]:
run_both('Can I take my data to another service provider?', marker='interoperable format')

,key,chunk,page,bi_score,ce_rank,ce_score,shift,snippet
bi_rank,,,,,,,,
1,,450,64,0.448,8,-10.09,#1 → #8,"ot be based on a provision in Article 45 or 46, including the provisions on binding corporate rules, and none …"
2,★,102,13,0.419,1,-4.95,#2 → #1,be changed. The fact that the processing of personal data is restricted should be clearly indicated in the sys…
3,,325,43,0.413,4,-7.74,#3 → #4,significance and the envisaged consequences of such processing for the data subject. 2. Where personal data ar…
4,,401,57,0.413,9,-10.40,#4 → #9,authorities and the communication of such personal data breaches to data subjects; (j) the transfer of persona…
5,,105,13,0.409,2,-5.78,#5 → #2,"particular, not imply the erasure of personal data concerning the data subject which have been provided by him…"
6,,7,2,0.401,6,-8.92,#6 → #6,authority in another Member State. (6) Rapid technological developments and globalisation have brought new cha…
7,,67,9,0.393,7,-9.29,#7 → #7,r the purposes of preventing fraud also constitutes a legitimate interest of the data controller concerned. Th…
8,,264,32,0.388,3,-5.95,#8 → #3,liability rules of intermediary service providers in Articles 12 to 15 of that Directive. Article 3 Territoria…
9,,153,19,0.378,5,-7.93,#9 → #5,"stablishment of certification mechanisms and data protection seals and marks should be encouraged, allowing da…"


Baseline top-3 : [450, 102, 325] 
Re-ranked top-3: [102, 105, 264]



=== ANSWER 1 (baseline) ===
Yes. Where the processing of personal data is carried out by automated means, the data subject is allowed to receive personal data concerning him or her (which he or she has provided to a controller) in a structured, commonly used, machine-readable, and interoperable format, and to transmit it to another controller [p.13]. 

**Conditions and exceptions:**
* This right applies where the data subject provided the personal data on the basis of his or her consent, or when the processing is necessary for the performance of a contract [p.13]. 
* The context does not state that it applies in other scenarios.



=== ANSWER 2 (re-ranked) ===
Yes, where the processing of personal data is carried out by automated means, you are allowed to receive the personal data concerning you—which you have provided to a controller—in a structured, commonly used, machine-readable, and interoperable format, and transmit it to another controller [p.13]. Data controllers should be encouraged to develop interoperable formats for data portability [p.13]. 

**Conditions and Exceptions:**
* **Conditions:** This right applies where you provided the personal data based on your consent, or where the processing is necessary for the performance of a contract [p.13]. 
* **Technical feasibility:** Where technically feasible, you also have the right to have the personal data transmitted directly from one controller to another [p.13].


**Observation (chunk 102: #2 → #1):** The control query remained correct in both pipelines: chunk 102, from Recital 68, moved from second to first without making the answer worse.

### 6.6 Rank-shift summary
Where the ★ key chunk sat before and after re-ranking (`None` = not in the top 10).

In [13]:
summary = []
for cat, q, marker in FAILURE_QUERIES:
    cands = retrieve(q)
    rr = [cid for cid, _ in rerank(q, cands)]
    key = [cid for cid, _ in cands if marker.lower() in chunks[cid]["text"].lower()]
    best = min(key, key=rr.index) if key else None
    summary.append({"category": cat, "query": q, "key_chunk": best,
                    "bi_rank": [c for c, _ in cands].index(best) + 1 if best is not None else None,
                    "ce_rank": rr.index(best) + 1 if best is not None else None})
pd.DataFrame(summary)

,category,query,key_chunk,bi_rank,ce_rank
0,Keyword trap,Can my employer monitor me at work?,562,3,1
1,Exception,"Can I get my personal data deleted, and when can the company refuse?",99,6,1
2,Negation,When does a company not need to tell affected people about a breach?,378,6,2
3,Buried chunk,Must every company appoint a data protection officer?,392,7,1
4,Control,Can I take my data to another service provider?,102,2,1


## 7. Latency profiling
Each stage is timed with `time.perf_counter`, after a warm-up, averaged over 20 runs of every failure query. LLM time comes from the real calls made in section 6, so no extra API calls are spent on it.

In [14]:
queries = [q for _, q, _ in FAILURE_QUERIES]
retrieve(queries[0]); rerank(queries[0], retrieve(queries[0]))   # warm-up

RUNS = 20
t_ret, t_rr = [], []
for _ in range(RUNS):
    for q in queries:
        t0 = time.perf_counter(); cands = retrieve(q)
        t1 = time.perf_counter(); rerank(q, cands)
        t2 = time.perf_counter()
        t_ret.append(t1 - t0); t_rr.append(t2 - t1)

ms = lambda xs: round(statistics.mean(xs) * 1000, 1)
# run_both calls the LLM baseline-first, so even entries = Answer 1 and odd entries = Answer 2
llm1, llm2 = ms(llm_times[0::2]), ms(llm_times[1::2])
pd.DataFrame([
    {"pipeline": "1 – Baseline",   "retrieval_ms": ms(t_ret), "rerank_ms": 0,         "retrieval_total_ms": ms(t_ret),         "llm_ms": llm1},
    {"pipeline": "2 – Re-ranking", "retrieval_ms": ms(t_ret), "rerank_ms": ms(t_rr), "retrieval_total_ms": ms(t_ret) + ms(t_rr), "llm_ms": llm2},
]).set_index("pipeline")

,retrieval_ms,rerank_ms,retrieval_total_ms,llm_ms
pipeline,,,,
1 – Baseline,6.6,0.0,6.6,1556.7
2 – Re-ranking,6.6,20.2,26.8,1757.4


In [15]:
# Scaling sanity check: what if we cross-encoded the WHOLE corpus for one query?
q = queries[0]
t0 = time.perf_counter()
cross_encoder.predict([(q, c["text"]) for c in chunks], batch_size=64)
full_ms = (time.perf_counter() - t0) * 1000
print(f"Cross-encoder over all {len(chunks)} chunks: {full_ms:.0f} ms "
      f"(~{full_ms / ms(t_rr):.0f}x the 10-candidate re-rank) for ONE query on a ~88-page document")

Cross-encoder over all 580 chunks: 894 ms (~44x the 10-candidate re-rank) for ONE query on a ~88-page document


## 8. Takeaways
- **Architecture:** The bi-encoder compresses an entire chunk into a single vector without directly comparing it with the query, so small but important differences, such as the word "not" or an exception to a rule, can easily be overlooked. The cross-encoder takes the query and a chunk together, letting each query token interact with the chunk's tokens through self-attention.
- **Rank shifts:** Re-ranking improved the ordering of passages that were already retrieved, particularly when the question depended on negation, exceptions or a specific legal condition. Baseline answers omitted essential clauses (queries 2 and 3) or misstated a rule (query 4). Re-ranking cannot recover information that the bi-encoder fails to retrieve among its candidates.
- **Latency:** The re-ranking stage added 20.2 ms, but compared with the full request time of approximately 1.6 seconds, the additional cost was only around 1–1.3%.
- **Scaling:** Cross-encoding all 580 chunks took 886 ms for a single query; at one million chunks this would take approximately 25 minutes per query. Document vectors can be precomputed, whereas query–document relevance scores must be calculated when needed, so a two-stage pipeline is more practical.

Full analysis: `report.md`.